# Lab 3b — LLM Mental Health Counselling Training App

**Course**: CSE 498
**Assignment**: Building an LLM Mental Health Training App (counsellors practise against simulated patients)
**Base codebase**: [efuayankey/aimes](https://github.com/efuayankey/aimes) — "AIMES" / "CALLM-CARE", authored by **Efua Yankey**
**My fork**: [shinnew9/aimes](https://github.com/shinnew9/aimes)
**My work for this lab**: branch [`feat/returning-patients-session-memory`](https://github.com/shinnew9/aimes/tree/feat/returning-patients-session-memory)
**Reviewable diff**: [compare view](https://github.com/shinnew9/aimes/compare/fix/spanish-english-translation-toggle...feat/returning-patients-session-memory)

---

## 0. Summary
This lab was **not** built from scratch. It was implemented as an extension of the AIMES counselling-training app already in use in the lab, rather than as a new standalone project.

I audited the existing codebase against the assignment brief, found that **two of the seven requirements had no implementation behind them**, and built the one with no code at all.

| Requirement | Already in the codebase | My work |
|---|---|---|
| Login / register | Yes (Firebase Auth) | — |
| Real vs. simulated patient split | Yes | — |
| New simulated-patient profile selection | **Partial** | Not addressed |
| **Choosing a patient spoken to before** | **No** | **Built** |
| Profile summary, patient speaks first | Yes | Extended |
| Multi-turn session with an explicit end | Yes | — |
| **Past-session summaries carried forward** | **No** | **Built** |
| Feedback on five dimensions + overall + suggestions | Yes | — |

**Size of the change**: 7 files, **1,034 lines added, 19 removed**. Nothing in the original codebase was rewritten or deleted.

**Key points**

1. Before this change, every session generated a new persona and retained nothing once the session ended, so two consecutive sessions with the same 
   counsellor were unrelated. The "existing patients" path in the brief therefore had nothing behind it.
2. A finished session is reduced to a **structured record of nine fields**, not a prose recap. The reasoning for that choice is in §3 and is the part 
   of this lab that required the most thought.
3. The record is used twice: shown to the counsellor **before** the next session, and included in the patient prompt **during** it.
4. The change adds **zero new type errors** against a stashed baseline, and the production build passes.

## 1. Starting Point — Auditing the Existing Codebase

The brief specifies a feature set. Before writing anything I went through the codebase to establish
what was already there, since duplicating working features would have been wasted effort.

The app is **Next.js 15 + TypeScript**, with **Firebase Authentication** and **Firestore**, and
**OpenAI `gpt-4o`** behind `src/app/api/*` routes. Roughly 21,500 lines of TypeScript.

| # | Requirement from the brief | Status found | Evidence |
|---|---|---|---|
| 1 | Home page with login / register | Implemented | `src/components/auth/` — `LoginForm`, `SignupForm`, `ForgotPasswordForm`, `AuthContext` |
| 2 | Second page: "real patients" / "simulated patients" | Implemented | `PatientModeSelector`, `PatientMode = 'real' \| 'simulated' \| 'cbt'` |
| 3a | New patient: gender, age group, ethnicity, **list of** concerns | **Partial** | `PatientGenerationOptions` accepts one `concern`; `ageRange` exists in the type but is not an explicit age-group choice |
| 3b | **Choose from patients spoken to before** | **Not implemented** | `generateRandomPatient()` is called on every session start; no patient is ever stored |
| 4 | Session page shows the profile; patient speaks first | Implemented | `showPatientInfo`, `generateOpeningMessage()` |
| 5 | Multi-turn until the counsellor ends the session | Implemented | Also has automatic end detection (`SessionEndDetectionService`), which exceeds the brief |
| 6 | **Store past sessions; show a summary next time** | **Not implemented** | See below |
| 7 | Feedback: cultural sensitivity, therapeutic progress, professional boundaries, empathy, conversational flow, overall, suggestions | Implemented | `ConversationFeedback.overallPerformance` has exactly those five plus `overallScore` |

### Why requirement 6 counted as "not implemented"
A method named `generateSessionSummary()` does exist, which initially looked like the feature. It is not:

```
sessionTopics:         keyword matching — if the text contains 'anxious'/'worried' → tag 'anxiety'
emotionalProgression:  simplified heuristic
duration, messageCount, averageResponseLength
```

It produces **keyword tags and message statistics**, and a search of the whole codebase for
`previousSession`, `sessionSummary`, `continuity`, `lastSession` and similar found **no path by
which its output reaches the next session's prompt**. It feeds the end-of-session detection UI only.

### Requirements 3b and 6 are one problem

A patient cannot be "existing" unless something about them persists, and a summary of a previous
session is meaningless without a patient to attach it to. They were treated as a single piece of work.

## 2. What Was Implemented

### 2.1 Patients persist between sessions

A generated persona is stored per counsellor in a new Firestore collection, `simulatedPatients`,
and **reused verbatim** on a return visit.

Reusing rather than regenerating matters more than it first appears: `generateRandomPatient()`
derives the backstory, personality traits and trust level from the profile options, so regenerating
from the same options would produce a *different person with the same demographics* every visit —
the opposite of continuity.

The patient is written **when the session starts**, not when it ends, so a session abandoned halfway
still leaves someone to come back to.

### 2.2 A finished session is distilled into a carry-forward record

At the end of a session the transcript is sent to `gpt-4o` through a new route,
`src/app/api/session-memory/route.ts`, and reduced to the nine fields described in §3.

Three properties of this step were treated as requirements rather than nice-to-haves:

- **No invention.** The prompt forbids recording any detail not present in the transcript.
- **Risk is never softened.** Risk indicators may not be omitted or rephrased away.
- **Failure must not lose data.** If extraction fails, a minimal locally-derived record is stored
  and the session is still saved. Losing the session because the summary failed would be worse than
  storing a thin summary.

### 2.3 The record is used in two places

| Where | What it does |
|---|---|
| **Before** the next session | The roster screen shows the previous summary, unresolved threads and any risk indicators, so the counsellor can prepare the way they would for a real returning client instead of rediscovering everything mid-conversation |
| **During** the next session | The record is appended to the patient's system prompt, rebuilt on every turn so it survives re-renders and is not dropped once the conversation gets long |

The continuity block is phrased as the patient's own recollection, and explicitly instructs against
reciting it:

> *Refer to these things the way a person does — in passing, half-remembered — never as a list or a
> recap. Do NOT summarise previous sessions. Do not say "last time we discussed...".*

Without that instruction the model opens a return visit by reading its own case file aloud.

## 3. Design Decision — What Counts as an "Important Fact"

The brief asks the student to *"think through what are important facts"* to store. This section is
that reasoning.

### Why not a prose summary

A paragraph summary is the obvious choice and was rejected. The problem is what happens downstream:
a paragraph can only be used one way — pasted whole into the next prompt. Once pasted, the model
treats it as material to narrate, and the patient opens the next session by summarising their own
previous session, which no real client does.

Discrete fields can be used **selectively and differently**:

- `rapportLevel` steers the greeting — it never appears in the prompt as text.
- `unresolvedThreads` steers the content.
- `riskFlags` are rendered in the UI by the application, **whether or not the model mentions them**.

A paragraph cannot be wired to three different consumers. The schema is the feature.

### The nine fields, and why each earns its place

The test applied to each candidate field was: *does this change how the next session should open?*
Anything that did not was left out.

| Field | Why it is kept |
|---|---|
| `headline` | One line, readable in the three seconds before a session starts |
| `presentingConcerns` | What was **actually** discussed, which routinely differs from the profile on file |
| `keyDisclosures` | Said for the first time. The patient will expect it to be remembered; asking again reads as not listening |
| `emotionalState` | Where the session **landed**, not where it started |
| `riskFlags` | Self-harm, hopelessness, safeguarding. The one field that must never be lost between sessions |
| `counselorCommitments` | "We agreed you'd try X." Unaddressed next time, the counsellor looks unreliable |
| `unresolvedThreads` | Gives the next session a place to begin |
| `culturalNotes` | Culture, faith, family and language factors that shaped the session |
| `rapportLevel` | Sets the patient's opening posture on return |

### Fields deliberately excluded

| Excluded | Reason |
|---|---|
| Message counts, duration, average response length | Already computed elsewhere, and they change nothing about how the next session opens |
| A diagnosis or clinical impression | The app is a training simulator, not a clinical record. Generating a diagnosis would invite it being read as one |
| Verbatim transcript quotes | The full transcript is stored anyway; duplicating it into the summary makes the prompt long without adding usable structure |

### How `rapportLevel` is consumed

This is the clearest example of a field doing work that prose could not:

| Recorded rapport | Opening behaviour on return |
|---|---|
| `low` | Guarded. "Hi." / "I'm here. I don't really know what else to say." |
| `medium` | Polite but testing. "Hi again. It's been a week, I guess." |
| `high` | Warmer and more direct. "Hey. I've actually been looking forward to this one." |

The opening line is chosen by the application from the recorded value, so the first turn of a return
visit reflects the previous session before the model is involved at all.

## 4. Verification
The repository ships with `typescript.ignoreBuildErrors` and `eslint.ignoreDuringBuilds` enabled in `next.config.ts`, so a passing build alone proves very little. Each check was therefore run **against a baseline** rather than in isolation.

| Check | Baseline (branch point) | After my changes | Verdict |
|---|---|---|---|
| `npx tsc --noEmit` | 74 errors | 74 errors | **0 new** |
| `npx eslint` (changed files) | 4 warnings, 0 errors | 4 warnings, 0 errors | **0 new** |
| `npm run build` | passes | passes, 12/12 pages | `/api/session-memory` registered |

### Method

The baseline was obtained by stashing the working tree (`git stash push -u`), re-running `tsc`, and restoring. The two errors reported inside `SimulatedSessionInterface.tsx` after my change are the **same two** present before it, at shifted line numbers:

| | Baseline line | After | Error |
|---|---|---|---|
| `ageRange` assignment | 804 | 1016 | TS2345 |
| `openings[concern]` index | 1226 | 1488 | TS7053 |

Both are pre-existing and untouched by this work.

### A bug found during verification
`resetSession()` cleared the session state but not the new continuity state. Pressing **New Session** after finishing one would therefore have carried the previous patient's memories into the next patient, who would have greeted the counsellor about a session that never happened. Fixed before the branch was pushed.

## 5. Environment Issues and Code Changes

| Issue | Resolution |
|---|---|
| **Node.js was not installed** on the server; `npx` and `npm` were unavailable, so nothing could be type-checked or built | Installed Node v20.18.1 under `~/.local/node` and put it on `PATH`. No root access needed |
| **74 pre-existing TypeScript errors.** `next.config.ts` sets `ignoreBuildErrors: true`, so they do not block a build and a naive run of `tsc` looks alarming | Established the count as a baseline before changing anything, and compared after (§4) |
| **`npm run build` failed at page-data collection** with *"Missing credentials… set the `OPENAI_API_KEY`"* — the API routes instantiate the OpenAI client at module scope | Pre-existing and unrelated to this work. Rebuilt with placeholder environment values to verify the build end to end |
| **Build then failed prerendering `/admin-ultra-secret-admin-2024`** with `auth/invalid-api-key`, because the Firebase config was absent | Also pre-existing. Supplied placeholder Firebase values; the build then completed 12/12 pages |
| **`firestore.rules` had no rule for `trainingSessions`** at all, although the app writes to it. Firestore denies by default when no rule matches | Added rules for both `trainingSessions` and the new `simulatedPatients`, scoped so a counsellor can only read and write their own records |
| **No `.env` template existed** in the repository, so required variables had to be read out of `src/config/env.ts` | Added `.env.local.example` listing every variable with empty values |

### Firestore rules added

Session summaries quote the transcript, so they are treated as the counsellor's private practice records rather than shared data:

```
match /simulatedPatients/{patientId} {
  allow create: if request.auth != null &&
    request.auth.uid == request.resource.data.counselorId;
  allow read, update, delete: if request.auth != null &&
    resource.data.counselorId == request.auth.uid;
}
```

## 6. Deviations from the Recommended Steps
The brief includes a suggested procedure (Phases 1–4) while stating that *"you need not follow too closely what these instructions say and can use your own approach."* The deviations are listed here for transparency.

| Recommended | Used | Reason |
|---|---|---|
| Clone `kevalsakhiya/chat-bots` as the starting point | **Not used** | The repository is **149 lines** in total: two standalone chatbot scripts with no login, no database, no pages, no personas and no feedback. The instructor directed this lab at the existing AIMES codebase (~21,500 lines), which already implements five of the seven requirements. Starting from the smaller repository would have meant rebuilding working features |
| Streamlit | **Next.js 15** | Determined by the existing codebase. (Note: `chat-bots` in fact uses **Chainlit**, not Streamlit) |
| Gemini (`gemini-1.5-pro` / `flash`), OpenAI optional | **OpenAI `gpt-4o`** | Every existing API route in the codebase uses `gpt-4o`. Introducing a second provider for one route would have made the feedback and summary paths inconsistent with each other. The brief permits ignoring OpenAI; it does not require Gemini |
| A sidebar model selector | **Not added** | Would only be meaningful with two providers wired up; see above |
| 3–4 preset personas in a dictionary / JSON file | **Already present** | `PatientSimulationService` generates personas from cultural background, gender, age range and concern, with backstory, communication style, emotional expression and trust level derived from them |

## 7. Deliverables

### Code
| Item | Location |
|---|---|
| Branch for this lab | [`feat/returning-patients-session-memory`](https://github.com/shinnew9/aimes/tree/feat/returning-patients-session-memory) |
| Reviewable diff | [compare view](https://github.com/shinnew9/aimes/compare/fix/spanish-english-translation-toggle...feat/returning-patients-session-memory) |
| Base codebase (Efua Yankey) | [efuayankey/aimes](https://github.com/efuayankey/aimes) |


### Files changed
| File | | Purpose |
|---|---|---|
| `src/app/api/session-memory/route.ts` | new | Extraction prompt and JSON schema for the carry-forward record |
| `src/services/sessionMemoryService.ts` | new | Extraction, normalisation of the model's output, continuity prompt block |
| `src/services/patientRosterService.ts` | new | Firestore persistence for saved patients |
| `src/components/counselor/ReturningPatientList.tsx` | new | The "existing patients" screen |
| `src/components/counselor/SimulatedSessionInterface.tsx` | modified | New / returning tabs, continuity wiring, end-of-session extraction |
| `src/types/SimulatedPatient.ts` | modified | `SessionMemory`, `StoredSimulatedPatient` |
| `firestore.rules` | modified | Rules for `simulatedPatients` and `trainingSessions` |


---

### A note on what this app is

Every "patient" in it is an AI-generated persona, not a real person, and nothing the app produces is
clinical advice. It exists so that counsellors in training can practise without practising on someone.